# CE888 – Lab 8: Learning from Sequences

**Module:** CE888 Data Science and Decision Making  
**University of Essex, CSEE**

---

## Lab Objectives

By the end of this lab you will be able to:

1. Explain how an RNN processes a sequence step-by-step
2. Manually compute RNN hidden states using NumPy
3. Count trainable parameters in SimpleRNN and LSTM layers
4. Pad variable-length sequences for batch training
5. Build a word-level tokenizer from scratch
6. Construct and compile stacked RNN and LSTM models in Keras

---

> **How to use this notebook:**  
> Read each section carefully, run every code cell, and complete the **✏️ Your Turn** exercises before moving on. These are not graded but directly prepare you for the Moodle CodeRunner questions.

## Imports

Run this cell first to import everything used throughout the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Keras / TensorFlow (only used in Section 5 and 6)
try:
    from tensorflow import keras
    from tensorflow.keras import layers
    print('✓ TensorFlow/Keras available:', keras.__version__)
except ImportError:
    print('⚠ TensorFlow not installed – Sections 1-4 will still work fine.')

print('✓ NumPy version:', np.__version__)

---
## Section 1 – What is a Sequence and Why Do RNNs Exist?

### 1.1 Sequences vs Independent Data Points

Tabular data (e.g. a spreadsheet of house prices) treats each row as independent — the order of rows doesn't matter.  
**Sequential data is different**: the *position* of each element carries meaning.

| Data Type | Example | Order Matters? |
|-----------|---------|----------------|
| Tabular | House price features | ❌ No |
| Image | Pixel grid | Partially |
| Time series | Stock prices over days | ✅ Yes |
| Text | Words in a sentence | ✅ Yes |

### 1.2 The Memory Problem with Dense Layers

A standard Dense (feedforward) layer processes its input in one shot — it has **no memory** of what it has seen before. If you feed it a sequence, it treats every position independently.

An **RNN** solves this by maintaining a **hidden state** `h_t` that acts as a running summary of everything seen so far. At each timestep the RNN:

1. Reads the new input `x_t`
2. Combines it with its current memory `h_{t-1}`
3. Produces a new memory `h_t`

```
 x_1 ──► [ RNN ] ──► h_1
               ↑
               h_0 = 0

 x_2 ──► [ RNN ] ──► h_2
               ↑
               h_1

 x_3 ──► [ RNN ] ──► h_3   ← final output used for prediction
               ↑
               h_2
```

In [ ]:
# ------------------------------------------------------------------
# Demo: What does a sequence look like as a NumPy array?
# ------------------------------------------------------------------

# A batch of 3 sequences, each 5 timesteps long, with 2 features per step
batch_size  = 3
timesteps   = 5
num_features = 2

np.random.seed(0)
X = np.random.randn(batch_size, timesteps, num_features)

print('Batch shape:', X.shape)             # (batch, timesteps, features)
print()
print('First sequence in the batch:')
print(X[0])  # shape (5, 2) — 5 timesteps, 2 features each
print()
print('First timestep of the first sequence:', X[0, 0])  # shape (2,)

---
## Section 2 – The RNN Forward Pass by Hand

### 2.1 The Core Equation

At timestep `t`, a SimpleRNN computes:

$$\mathbf{h}_t = \tanh(\mathbf{W}_x \cdot \mathbf{x}_t + \mathbf{W}_h \cdot \mathbf{h}_{t-1} + \mathbf{b})$$

| Term | Shape | Meaning |
|------|-------|---------|
| `x_t` | `(F,)` | Input vector at step `t` |
| `h_{t-1}` | `(H,)` | Hidden state from step `t-1` |
| `W_x` | `(H, F)` | Input weight matrix |
| `W_h` | `(H, H)` | Recurrent (hidden-to-hidden) weight matrix |
| `b` | `(H,)` | Bias vector |
| `h_t` | `(H,)` | New hidden state |

`F` = input features, `H` = hidden units

### 2.2 The `@` Operator

`@` in Python is the **matrix multiplication operator** (introduced in Python 3.5).  
For a matrix `A` of shape `(m, n)` and a vector `v` of shape `(n,)`, `A @ v` gives a vector of shape `(m,)`.

```python
W_x @ x_t      # same as np.dot(W_x, x_t)
W_h @ h_prev   # same as np.dot(W_h, h_prev)
```

In [ ]:
# ------------------------------------------------------------------
# Demo: @ operator for matrix-vector multiplication
# ------------------------------------------------------------------

W = np.array([[1, 0],   # shape (3, 2)
              [0, 1],
              [1, 1]])

x = np.array([3, 4])    # shape (2,)

result_at  = W @ x
result_dot = np.dot(W, x)

print('W @ x        :', result_at)    # [3, 4, 7]
print('np.dot(W, x) :', result_dot)   # [3, 4, 7]  ← identical
print()
print('W * x (WRONG - element-wise):')
print(W * x)   # completely different!

In [ ]:
# ------------------------------------------------------------------
# Worked Example: RNN forward pass for a 4-step sequence
# ------------------------------------------------------------------

np.random.seed(42)

T = 4    # timesteps
F = 3    # input features
H = 5    # hidden units

# Random weights (in a real network these would be learned)
W_x = np.random.randn(H, F) * 0.1
W_h = np.random.randn(H, H) * 0.1
b   = np.zeros(H)

# Random input sequence — shape (T, F)
input_sequence = np.random.randn(T, F)

# ---- Forward pass ----
h_t = np.zeros(H)   # h_0 initialised to zeros
print(f'h_0 (initial state): {h_t}\n')

for t, x_t in enumerate(input_sequence):
    h_t = np.tanh(W_x @ x_t + W_h @ h_t + b)
    print(f'h_{t+1} = {np.round(h_t, 4)}')

print(f'\nFinal hidden state used for prediction: {np.round(h_t, 4)}')

**Observations from the output above:**

- All values are between −1 and 1 because of **tanh**.
- Each `h_t` depends on both the current input AND all previous inputs (via the recurrence).
- The final `h_T` is a compressed summary of the whole sequence — this is what gets passed to a Dense layer for prediction.

In [ ]:
# ✏️ YOUR TURN 1  (Prepares for Graded Q1)
# -------------------------------------------------------
# Complete the function below so it:
#   - Initialises h_0 as a zero vector of length H
#   - Loops over every timestep in input_sequence
#   - Updates h_t using the RNN equation with tanh
#   - Appends each h_t to a list and returns it

def rnn_forward(input_sequence, W_x, W_h, b):
    """
    Returns a list of hidden states [h_1, h_2, ..., h_T]
    """
    # TODO: initialise h_t
    # TODO: loop and update
    pass


# Quick test — should print 4 states each of shape (5,)
np.random.seed(42)
seq = np.random.randn(4, 3)
Wx  = np.random.randn(5, 3) * 0.1
Wh  = np.random.randn(5, 5) * 0.1
bias = np.zeros(5)

states = rnn_forward(seq, Wx, Wh, bias)
if states is not None:
    print(f'Number of states returned: {len(states)}')
    print(f'Shape of each state:       {states[0].shape}')
    all_bounded = all(np.all(s >= -1) and np.all(s <= 1) for s in states)
    print(f'All values in [-1, 1]:     {all_bounded}')

---
## Section 3 – Counting Trainable Parameters

### 3.1 Why Does Parameter Count Matter?

- **More parameters** → more expressive model → needs more data and longer training
- **Fewer parameters** → simpler model → faster, but may underfit
- Always sanity-check your model's parameter count before training

### 3.2 SimpleRNN Parameters

A SimpleRNN layer with `F` input features and `H` hidden units has **three** sets of parameters:

| Matrix | Shape | Count |
|--------|-------|-------|
| `W_x` (input weights) | `(H, F)` | `H × F` |
| `W_h` (recurrent weights) | `(H, H)` | `H × H` |
| `b` (bias) | `(H,)` | `H` |
| **Total** | | **`H×F + H×H + H`** |

### 3.3 LSTM Parameters

An LSTM has **four gates** (forget, input, cell, output). Each gate has the same structure as a SimpleRNN cell:

$$\text{LSTM params} = 4 \times (H \times F + H \times H + H)$$

### 3.4 Stacked Layers

When stacking, each layer after the first receives the **previous layer's hidden state** as its input:

```
Input     → [RNN H=32] → [RNN H=16] → [RNN H=8] → Dense
F features    input=F     input=32      input=16
```

In [ ]:
# ------------------------------------------------------------------
# Demo: Computing parameter counts
# ------------------------------------------------------------------

def simple_rnn_params(input_features, hidden_units):
    """Parameter count for a single SimpleRNN layer."""
    F, H = input_features, hidden_units
    return F * H + H * H + H

def lstm_params(input_features, hidden_units):
    """Parameter count for a single LSTM layer."""
    return 4 * simple_rnn_params(input_features, hidden_units)


# Example: F=3 input features, H=8 hidden units
F, H = 3, 8
rnn_p  = simple_rnn_params(F, H)
lstm_p = lstm_params(F, H)

print(f'SimpleRNN ({F} → {H}): {rnn_p} parameters')
print(f'  W_x: {F} × {H} = {F*H}')
print(f'  W_h: {H} × {H} = {H*H}')
print(f'  b:   {H}')
print()
print(f'LSTM ({F} → {H}): {lstm_p} parameters  (= 4 × {rnn_p})')
print(f'Ratio LSTM/RNN: {lstm_p / rnn_p:.1f}')

In [ ]:
# ------------------------------------------------------------------
# Demo: Parameter count for a stacked RNN
# ------------------------------------------------------------------

input_features = 4
units_list     = [16, 12, 8]

total = 0
current_input = input_features

print(f'Input features: {input_features}')
print(f'Layers: {units_list}\n')

for i, H in enumerate(units_list):
    F = current_input
    params = F * H + H * H + H
    print(f'Layer {i+1}: input={F}, hidden={H}')
    print(f'  W_x={F}×{H}={F*H}, W_h={H}×{H}={H*H}, b={H}')
    print(f'  Layer params: {params}')
    total += params
    current_input = H   # next layer's input = this layer's output

print(f'\nTotal RNN params: {total}')

In [ ]:
# ✏️ YOUR TURN 2  (Prepares for Graded Q6 and Q8)
# -------------------------------------------------------
# 2a) Implement count_rnn_params for a stacked RNN:

def count_rnn_params(input_features, units_list):
    """
    Returns the total parameter count across all SimpleRNN layers.
    """
    # TODO
    pass

# Test: should be 852
result = count_rnn_params(4, [16, 12, 8])
print('Stacked [16,12,8] with F=4:', result, '← expected 852')

# -------------------------------------------------------
# 2b) Complete compare_rnn_lstm_params:

def compare_rnn_lstm_params(input_features, hidden_units):
    """
    Returns dict with keys: rnn_params, lstm_params, ratio, lstm_extra_params
    """
    # TODO
    pass

# Test: ratio should always be 4.0
d = compare_rnn_lstm_params(3, 8)
if d:
    print(f'\nRNN params:  {d["rnn_params"]}')
    print(f'LSTM params: {d["lstm_params"]}')
    print(f'Ratio:       {d["ratio"]}')

---
## Section 4 – Padding Sequences and Output Shapes

### 4.1 Why Padding?

Neural networks process data in **batches** — all samples in a batch must have the same shape. Real sequences (e.g. sentences) have different lengths. We fix this by:

- **Padding**: adding zeros to short sequences to reach `maxlen`
- **Truncating**: cutting long sequences down to `maxlen`

Both can happen at the **start** or **end** of the sequence. Using the **end** (`'post'`) is most common for text.

```
maxlen = 5

Short:  [1, 2]          → [1, 2, 0, 0, 0]   (post-pad)
Long:   [1, 2, 3, 4, 5, 6, 7] → [1, 2, 3, 4, 5]   (post-truncate)
Exact:  [1, 2, 3, 4, 5] → [1, 2, 3, 4, 5]   (unchanged)
```

### 4.2 RNN Output Shape Rules

| `return_sequences` | Output shape |
|---|---|
| `False` (default) | `(batch_size, hidden_units)` |
| `True` | `(batch_size, timesteps, hidden_units)` |

In [ ]:
# ------------------------------------------------------------------
# Demo: pad_sequences
# ------------------------------------------------------------------

from tensorflow.keras.preprocessing.sequence import pad_sequences

sequences = [
    [1, 2],
    [3, 4, 5, 6, 7],
    [8, 9, 10]
]

maxlen = 5
padded = pad_sequences(sequences, maxlen=maxlen, padding='post', truncating='post')

print('Original sequences:')
for s in sequences:
    print(' ', s)
print()
print(f'After padding/truncating to maxlen={maxlen}:')
print(padded)
print('Shape:', padded.shape)  # (3, 5)

In [ ]:
# ------------------------------------------------------------------
# Demo: RNN output shapes — return_sequences=False vs True
# ------------------------------------------------------------------

def get_rnn_output_shape(batch_size, timesteps, input_features, hidden_units, return_sequences):
    """Pure-Python output shape calculator (no Keras needed)."""
    if return_sequences:
        return (batch_size, timesteps, hidden_units)
    else:
        return (batch_size, hidden_units)


batch, T, F, H = 32, 10, 5, 16

shape_last = get_rnn_output_shape(batch, T, F, H, return_sequences=False)
shape_full = get_rnn_output_shape(batch, T, F, H, return_sequences=True)

print(f'Input shape:                    ({batch}, {T}, {F})')
print(f'return_sequences=False output:  {shape_last}   ← only last hidden state')
print(f'return_sequences=True  output:  {shape_full}   ← all T hidden states')

In [ ]:
# ✏️ YOUR TURN 3  (Prepares for Graded Q2 and Q5)
# -------------------------------------------------------

# 3a) Implement prepare_sequences using pad_sequences

def prepare_sequences(sequences, maxlen):
    """
    Pads/truncates sequences using post-padding and post-truncation.
    Returns a 2D NumPy array of shape (len(sequences), maxlen).
    """
    # TODO
    pass

test_seqs = [[1, 2, 3], [4, 5], [6, 7, 8, 9]]
result    = prepare_sequences(test_seqs, maxlen=4)
if result is not None:
    print('Padded output:', result.shape)   # should be (3, 4)
    print(result)

# -------------------------------------------------------
# 3b) Without using Keras, predict the output shape of:
#     A SimpleRNN with 64 units, batch_size=16, timesteps=20
#     when return_sequences=True vs False

# Write your answers as comments:
# return_sequences=False: shape = ???
# return_sequences=True:  shape = ???

---
## Section 5 – Word-Level Text Tokenization from Scratch

### 5.1 The Text → Numbers Pipeline

RNNs and LSTMs cannot work directly on raw text strings. We must convert text to numbers first:

```
"I love machine learning"
       ↓  lowercase + split
["i", "love", "machine", "learning"]
       ↓  index lookup
[3, 7, 1, 12]
       ↓  pad/truncate to maxlen
[3, 7, 1, 12, 0, 0]    (maxlen=6)
       ↓  Embedding layer
[[...], [...], [...], [...], [...], [...]]  (dense vectors)
       ↓  LSTM
h_T   (final hidden state for classification)
```

### 5.2 Building a Vocabulary

We scan all training sentences and assign an integer to each **unique word** in order of first appearance.  
Index `0` is reserved for **`<UNK>`** (unknown words seen only at inference time).

```python
corpus = ["the cat sat", "the dog ran"]
# After fitting:
# word_to_index = {'the': 1, 'cat': 2, 'sat': 3, 'dog': 4, 'ran': 5}
```

### 5.3 Transforming New Sentences

```python
# Known words → their index
# Unknown words → 0
transform("the cat flew") → [1, 2, 0]
#                                   ↑ 'flew' was not in training corpus
```

In [ ]:
# ------------------------------------------------------------------
# Demo: A minimal word tokenizer built step-by-step
# ------------------------------------------------------------------

corpus = [
    "The quick brown fox",
    "The fox jumps over the lazy dog"
]

# Step 1: Build vocabulary
word_to_index = {}
idx = 1   # 0 is reserved for <UNK>

for sentence in corpus:
    for word in sentence.lower().split():
        if word not in word_to_index:
            word_to_index[word] = idx
            idx += 1

print('Vocabulary:')
for word, index in word_to_index.items():
    print(f'  {word!r:12} → {index}')

print(f'\nVocabulary size: {len(word_to_index)} words')

In [ ]:
# Step 2: Transform sentences
test_sentences = [
    "the quick brown fox",   # all known words
    "the cat sat",           # 'cat' and 'sat' are unknown
]

for sentence in test_sentences:
    tokens = [
        word_to_index.get(word.lower(), 0)   # 0 for unknown
        for word in sentence.split()
    ]
    print(f'{sentence!r:35} → {tokens}')

In [ ]:
# ✏️ YOUR TURN 4  (Prepares for Graded Q7)
# -------------------------------------------------------
# Wrap the tokenizer logic into a class with fit() and transform() methods.

class WordTokenizer:

    def __init__(self):
        # TODO: initialise word_to_index
        pass

    def fit(self, sentences):
        """
        Build vocabulary from a list of sentences.
        Words are lowercased. Index 0 reserved for <UNK>.
        Returns self.
        """
        # TODO
        pass

    def transform(self, sentences):
        """
        Convert sentences to lists of integer indices.
        Unknown words → 0.
        Returns list of lists of ints.
        """
        # TODO
        pass


# Quick tests
tok = WordTokenizer()
tok.fit(["hello world", "hello python"])
print('Vocabulary:', tok.word_to_index)   # {'hello':1, 'world':2, 'python':3}

result = tok.transform(["hello world", "hello unknown"])
print('Transformed:', result)             # [[1, 2], [1, 0]]

---
## Section 6 – Building Keras Models

### 6.1 Stacked SimpleRNN for Regression

The **critical rule** when stacking recurrent layers:

> All layers **except the last** must use `return_sequences=True`  
> so that the next layer receives a sequence to process.

```
Input (batch, T, F)
    ↓
SimpleRNN(32, return_sequences=True)   → (batch, T, 32)   ← outputs a sequence
    ↓
SimpleRNN(16, return_sequences=True)   → (batch, T, 16)   ← outputs a sequence
    ↓
SimpleRNN(8,  return_sequences=False)  → (batch, 8)        ← last state only
    ↓
Dense(1)                               → (batch, 1)        ← regression output
```

### 6.2 LSTM for Text Classification

```
Input (batch, maxlen)           ← integer word indices
    ↓
Embedding(vocab_size, embed_dim)  → (batch, maxlen, embed_dim)
    ↓
LSTM(128)                         → (batch, 128)
    ↓
Dropout(0.5)
    ↓
Dense(1, activation='sigmoid')    → (batch, 1)   ← probability
```

### 6.3 Dropout in RNNs

⚠️ **Do NOT** use a standalone `Dropout` layer between stacked recurrent layers.  
Use the built-in `dropout` and `recurrent_dropout` arguments instead:

```python
layers.LSTM(128, dropout=0.2, recurrent_dropout=0.2)
```

A `Dropout` layer placed **after the final RNN layer** (before Dense) is fine.

In [ ]:
# ------------------------------------------------------------------
# Demo: Stacked SimpleRNN model
# ------------------------------------------------------------------

from tensorflow import keras
from tensorflow.keras import layers

timesteps    = 20
num_features = 3

model = keras.Sequential([
    layers.SimpleRNN(32, return_sequences=True,
                     input_shape=(timesteps, num_features)),  # → (batch, 20, 32)
    layers.SimpleRNN(16, return_sequences=True),              # → (batch, 20, 16)
    layers.SimpleRNN(8,  return_sequences=False),             # → (batch, 8)
    layers.Dense(1)                                           # → (batch, 1)
])

model.compile(optimizer='adam', loss='mse')
model.summary()

In [ ]:
# Verify return_sequences settings
rnn_layers = [l for l in model.layers if isinstance(l, layers.SimpleRNN)]

print('Layer-by-layer return_sequences check:')
for i, layer in enumerate(rnn_layers):
    expected = (i < len(rnn_layers) - 1)  # True for all but last
    status   = '✓' if layer.return_sequences == expected else '✗ WRONG!'
    print(f'  Layer {i+1} ({layer.name}): return_sequences={layer.return_sequences}  {status}')

In [ ]:
# ------------------------------------------------------------------
# Demo: LSTM text classifier
# ------------------------------------------------------------------

vocab_size = 5000
embed_dim  = 64
lstm_units = 128
max_len    = 200

lstm_model = keras.Sequential([
    layers.Embedding(input_dim=vocab_size,
                     output_dim=embed_dim,
                     input_length=max_len),
    layers.LSTM(lstm_units, return_sequences=False),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='rmsprop',
                   loss='binary_crossentropy',
                   metrics=['accuracy'])

lstm_model.summary()

In [ ]:
# ------------------------------------------------------------------
# Understanding the Embedding layer
# ------------------------------------------------------------------

# An Embedding layer is a learnable lookup table:
# integer index → dense vector of size embed_dim

# Its parameter count is simply:
emb_params = vocab_size * embed_dim
print(f'Embedding parameters: {vocab_size} × {embed_dim} = {emb_params:,}')
print('(One vector learned per word in the vocabulary)')

# Demonstrate the shape transformation:
import numpy as np
dummy_input = np.random.randint(0, vocab_size, size=(4, max_len))  # batch of 4
embedding_output = lstm_model.layers[0](dummy_input)
print(f'\nEmbedding input shape:  {dummy_input.shape}')        # (4, 200)
print(f'Embedding output shape: {embedding_output.shape}')    # (4, 200, 64)

In [ ]:
# ✏️ YOUR TURN 5  (Prepares for Graded Q3 and Q4)
# -------------------------------------------------------

# 5a) Implement build_stacked_rnn
def build_stacked_rnn(timesteps, num_features, units_list):
    """
    Returns a compiled Sequential model:
    - SimpleRNN layers from units_list
    - return_sequences=True for all but the last RNN layer
    - Dense(1) output
    - Compiled with adam + mse
    """
    # TODO
    pass

m = build_stacked_rnn(10, 4, [32, 16])
if m:
    print('Stacked RNN layers:', len(m.layers))
    m.summary()

# -------------------------------------------------------
# 5b) Implement build_lstm_classifier
def build_lstm_classifier(vocab_size, embed_dim, lstm_units, max_len):
    """
    Returns a compiled Sequential model:
    Embedding → LSTM → Dropout(0.5) → Dense(1, sigmoid)
    Compiled with rmsprop + binary_crossentropy + accuracy
    """
    # TODO
    pass

m2 = build_lstm_classifier(1000, 32, 64, 100)
if m2:
    print('\nLSTM classifier layers:', len(m2.layers))
    m2.summary()

---
## Section 7 – SimpleRNN vs LSTM vs GRU: Side-by-Side Comparison

| | SimpleRNN | LSTM | GRU |
|---|---|---|---|
| **Gating** | None | 3 gates + cell state | 2 gates |
| **Parameters** | `H²+HF+H` | `4×(H²+HF+H)` | `3×(H²+HF+H)` |
| **Vanishing gradient** | Severe | Solved | Solved |
| **Training speed** | Fastest | Slowest | Middle |
| **Best for** | Short sequences / teaching | Long sequences, NLP | Limited data, speed matters |
| **Keras** | `layers.SimpleRNN` | `layers.LSTM` | `layers.GRU` |

In [ ]:
# ------------------------------------------------------------------
# Visualise: How parameter counts scale with hidden units
# ------------------------------------------------------------------

import matplotlib.pyplot as plt
import numpy as np

F = 10   # fixed input features
hidden_units = np.arange(4, 257, 4)

rnn_params  = (F * hidden_units) + (hidden_units**2) + hidden_units
gru_params  = 3 * rnn_params
lstm_params = 4 * rnn_params

plt.figure(figsize=(9, 5))
plt.plot(hidden_units, rnn_params,  label='SimpleRNN', linewidth=2)
plt.plot(hidden_units, gru_params,  label='GRU (3×)',   linewidth=2, linestyle='--')
plt.plot(hidden_units, lstm_params, label='LSTM (4×)',  linewidth=2, linestyle=':')
plt.xlabel('Hidden Units (H)', fontsize=12)
plt.ylabel('Number of Parameters', fontsize=12)
plt.title(f'Parameter Count vs Hidden Units (Input Features F={F})', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'At H=128, F=10:')
H = 128
rnn_p = F*H + H*H + H
print(f'  SimpleRNN: {rnn_p:,} params')
print(f'  GRU:       {3*rnn_p:,} params')
print(f'  LSTM:      {4*rnn_p:,} params')

---
## Section 8 – Summary and Pre-Assessment Checklist

Before moving to Moodle CodeRunner, verify you can answer **yes** to each item below.

### ✅ Concept Checklist

- [ ] I can explain what `h_t = tanh(W_x @ x_t + W_h @ h_{t-1} + b)` computes
- [ ] I know what `@` does and how it differs from `*`
- [ ] I know why `h_0` is initialised to zeros
- [ ] I can calculate the parameter count for a SimpleRNN layer given `F` and `H`
- [ ] I know why LSTM has 4× more parameters than SimpleRNN
- [ ] I can compute parameters for a stacked RNN (tracking `current_input`)
- [ ] I know when to use `return_sequences=True` vs `False`
- [ ] I understand what padding does and why we need it
- [ ] I can build a word-to-index vocabulary and handle unknown words with index 0
- [ ] I can construct a stacked SimpleRNN model with correct `return_sequences` flags
- [ ] I can construct an LSTM classifier with Embedding + Dropout layers

In [ ]:
# ------------------------------------------------------------------
# Final self-test: Run all your implementations together
# ------------------------------------------------------------------

import numpy as np
print('=' * 55)
print('  FINAL SELF-TEST')
print('=' * 55)

tests_passed = 0

# --- Test rnn_forward ---
try:
    np.random.seed(0)
    states = rnn_forward(np.random.randn(5, 3),
                         np.random.randn(4, 3) * 0.1,
                         np.random.randn(4, 4) * 0.1,
                         np.zeros(4))
    assert len(states) == 5 and states[0].shape == (4,)
    assert all(np.all(s >= -1) and np.all(s <= 1) for s in states)
    print('✓ rnn_forward          – PASSED')
    tests_passed += 1
except Exception as e:
    print(f'✗ rnn_forward          – FAILED: {e}')

# --- Test count_rnn_params ---
try:
    assert count_rnn_params(4, [16, 12, 8]) == 852
    assert count_rnn_params(3, [8]) == 96
    print('✓ count_rnn_params     – PASSED')
    tests_passed += 1
except Exception as e:
    print(f'✗ count_rnn_params     – FAILED: {e}')

# --- Test compare_rnn_lstm_params ---
try:
    d = compare_rnn_lstm_params(3, 8)
    assert d['rnn_params'] == 96
    assert d['lstm_params'] == 384
    assert d['ratio'] == 4.0
    assert d['lstm_extra_params'] == 288
    print('✓ compare_rnn_lstm     – PASSED')
    tests_passed += 1
except Exception as e:
    print(f'✗ compare_rnn_lstm     – FAILED: {e}')

# --- Test WordTokenizer ---
try:
    tok = WordTokenizer()
    tok.fit(['hello world', 'hello python'])
    result = tok.transform(['hello world', 'hello unknown'])
    assert result == [[1, 2], [1, 0]], f'Got {result}'
    print('✓ WordTokenizer        – PASSED')
    tests_passed += 1
except Exception as e:
    print(f'✗ WordTokenizer        – FAILED: {e}')

# --- Test prepare_sequences ---
try:
    out = prepare_sequences([[1, 2], [3, 4, 5, 6, 7]], maxlen=4)
    assert out.shape == (2, 4)
    assert out[0, 2] == 0  # padded
    assert out[1, 3] == 7  # NOT truncated from start — post means keep first 4
    print('✓ prepare_sequences    – PASSED')
    tests_passed += 1
except Exception as e:
    print(f'✗ prepare_sequences    – FAILED: {e}')

# --- Test build_stacked_rnn ---
try:
    m = build_stacked_rnn(10, 3, [16, 8])
    assert len(m.layers) == 3
    rnn_ls = [l for l in m.layers if isinstance(l, layers.SimpleRNN)]
    assert rnn_ls[0].return_sequences == True
    assert rnn_ls[1].return_sequences == False
    print('✓ build_stacked_rnn    – PASSED')
    tests_passed += 1
except Exception as e:
    print(f'✗ build_stacked_rnn    – FAILED: {e}')

# --- Test build_lstm_classifier ---
try:
    m2 = build_lstm_classifier(1000, 32, 64, 50)
    assert len(m2.layers) == 4
    assert isinstance(m2.layers[0], layers.Embedding)
    assert isinstance(m2.layers[1], layers.LSTM)
    dummy = np.random.randint(0, 1000, (5, 50))
    preds = m2.predict(dummy, verbose=0)
    assert preds.shape == (5, 1)
    assert np.all(preds >= 0) and np.all(preds <= 1)
    print('✓ build_lstm_classifier – PASSED')
    tests_passed += 1
except Exception as e:
    print(f'✗ build_lstm_classifier – FAILED: {e}')

print('=' * 55)
print(f'Results: {tests_passed}/7 tests passed')
if tests_passed == 7:
    print('🎉 All tests passed! You are ready for Moodle.')
else:
    print('⚠ Review the failed sections before attempting the graded questions.')

---
## 🚀 Next Steps

1. If all 7 self-tests pass → head to **Moodle CodeRunner** and attempt the 8 graded questions.
2. If some tests fail → re-read the relevant section and fix your implementation.
3. For extra challenge, try:
   - Adding `dropout` and `recurrent_dropout` to the LSTM model and observe if validation accuracy improves.
   - Experimenting with `GRU` instead of `LSTM` — change one word in the code and compare `model.count_params()`.
   - Extending `WordTokenizer` to support n-gram tokenization.

---

*CE888 – University of Essex, CSEE | Dr. Ana Matran-Fernandez*